In [1]:
"""
Build assumption_master sheet: conversion_option
-----
Extracts Conversion Option (CO) rates and structure per product.

Products with CO in scope: gla, lsdb, ci, spouse_gla, spouse_lsdb

Schema: product | age_condition | co_type | rate | notes
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib, csv, openpyxl, polars as pl

BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE_XLSX = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "Assumptions"

ASSUMPTION_VERSION = "A2026_07"
SNAPSHOT_DIR = BASE / "assumption_snapshots" / f"assumption_{ASSUMPTION_VERSION}"
MASTER_XLSX = BASE / "assumption_master" / f"assumption_master_{ASSUMPTION_VERSION}.xlsx"
BUILD_LOG = SNAPSHOT_DIR / "build_log.csv"

SHEET_NAME = "conversion_option"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")

Building: conversion_option


In [2]:
def find_all_cells_any_col(ws, label, max_row=1200, max_col=30):
    hits = []
    for row in range(1, max_row + 1):
        for col in range(1, max_col + 1):
            v = ws.cell(row=row, column=col).value
            if isinstance(v, str) and v.strip() == label:
                hits.append((row, col))
    return hits

def get_col_letter(col):
    return openpyxl.utils.get_column_letter(col)

def peek(ws, row, col, span=4):
    parts = []
    for dc in range(0, span + 1):
        v = ws.cell(row=row, column=col + dc).value
        parts.append(f"{get_col_letter(col + dc)}{row}={v!r}")
    return " | ".join(parts)

# Locate every CO-related anchor across all product blocks
DIAGNOSTIC_LABELS = [
    # GLA CO block
    "GLA CO",
    "CO Structure",
    "Retirement and Withdrawal",
    "Withdrawal only",
    "<=65 & Retirement and Withdrawal",
    "<=65 & Retirement or Withdrawal only",
    ">65 & Retirement and Withdrawal",
    ">65 & Retirement or Withdrawal only",
    # LSDB CO
    "LSDB CO",
    # CI CO
    "CI CO",
    # Spouse CO
    "SP CO",
    # Age condition labels (used across CO blocks)
    "<=65",
    ">65",
]

print("=" * 90)
print("DIAGNOSTIC — CO anchor locations")
print("=" * 90)

for label in DIAGNOSTIC_LABELS:
    hits = find_all_cells_any_col(ws, label)
    print(f"\n'{label}': {len(hits)} match(es)")
    for i, (r, c) in enumerate(hits[:6]):
        print(f"  [{i}] {peek(ws, r, c)}")

DIAGNOSTIC — CO anchor locations

'GLA CO': 1 match(es)
  [0] F214='GLA CO' | G214=None | H214=None | I214='Load automatically' | J214=1

'CO Structure': 1 match(es)
  [0] F210='CO Structure' | G210=None | H210=None | I210=None | J210=None

'Retirement and Withdrawal': 1 match(es)
  [0] F211='Retirement and Withdrawal' | G211=1 | H211=None | I211=None | J211=None

'Withdrawal only': 1 match(es)
  [0] F212='Withdrawal only' | G212=2 | H212=None | I212=None | J212=None

'<=65 & Retirement and Withdrawal': 1 match(es)
  [0] F215='<=65 & Retirement and Withdrawal' | G215=0.037 | H215=None | I215='Enter own loadings' | J215=2

'<=65 & Retirement or Withdrawal only': 1 match(es)
  [0] F216='<=65 & Retirement or Withdrawal only' | G216=0.02 | H216=None | I216='No loadings' | J216=3

'>65 & Retirement and Withdrawal': 1 match(es)
  [0] F217='>65 & Retirement and Withdrawal' | G217=0.037 | H217=None | I217='Active Selection' | J217=1

'>65 & Retirement or Withdrawal only': 1 match(es)
  [0] F21

In [3]:
all_rows: list[dict] = []

# =============================================================================
# GLA CO — full grid (age × co_type)
# =============================================================================
# Row layout:
#   F215 = '<=65 & Retirement and Withdrawal',  G215 = 0.037
#   F216 = '<=65 & Retirement or Withdrawal only', G216 = 0.02
#   F217 = '>65 & Retirement and Withdrawal',    G217 = 0.037
#   F218 = '>65 & Retirement or Withdrawal only', G218 = 0.02

GLA_CO_GRID = [
    (215, "<=65", "retirement_and_withdrawal", "GLA CO rate: age ≤65, ret+wd"),
    (216, "<=65", "withdrawal_only",           "GLA CO rate: age ≤65, wd only"),
    (217, ">65",  "retirement_and_withdrawal", "GLA CO rate: age >65, ret+wd"),
    (218, ">65",  "withdrawal_only",           "GLA CO rate: age >65, wd only"),
]

for row, age_cond, co_type, notes in GLA_CO_GRID:
    label = ws.cell(row=row, column=6).value  # col F
    rate  = ws.cell(row=row, column=7).value  # col G
    all_rows.append({
        "product": "gla",
        "age_condition": age_cond,
        "co_type": co_type,
        "rate": rate,
        "source_label": label,
        "source_row": row,
        "notes": notes,
    })

# =============================================================================
# GLA CO Type Code map (Retirement+Withdrawal → 1, Withdrawal only → 2)
# Pricing engine uses this to resolve CO type text → numeric index.
# Stored as separate rows with co_type acting as key and rate holding the code.
# =============================================================================
GLA_CO_TYPE_MAP = [
    (211, "retirement_and_withdrawal", "GLA CO Structure code: Retirement+Withdrawal → 1"),
    (212, "withdrawal_only",           "GLA CO Structure code: Withdrawal only → 2"),
]

for row, co_type, notes in GLA_CO_TYPE_MAP:
    label = ws.cell(row=row, column=6).value  # col F
    code  = ws.cell(row=row, column=7).value  # col G
    all_rows.append({
        "product": "gla",
        "age_condition": "code_map",   # sentinel — not a rate row
        "co_type": co_type,
        "rate": code,
        "source_label": label,
        "source_row": row,
        "notes": notes,
    })

# =============================================================================
# LSDB CO (rows 267-268)
# =============================================================================
LSDB_CO = [
    (267, "<=65", 0.05, "LSDB CO rate: age ≤65"),
    (268, ">65",  0.05, "LSDB CO rate: age >65"),
]

for row, age_cond, expected, notes in LSDB_CO:
    label = ws.cell(row=row, column=2).value  # col B
    rate  = ws.cell(row=row, column=3).value  # col C
    all_rows.append({
        "product": "lsdb",
        "age_condition": age_cond,
        "co_type": "all",
        "rate": rate,
        "source_label": f"LSDB CO {label}",
        "source_row": row,
        "notes": notes,
    })

# =============================================================================
# CI CO (row 335) — single rate, no >65 split
# =============================================================================
CI_CO = [(335, "<=65", "CI CO rate: age ≤65 (source has no >65 row)")]

for row, age_cond, notes in CI_CO:
    label = ws.cell(row=row, column=5).value  # col E
    rate  = ws.cell(row=row, column=6).value  # col F
    all_rows.append({
        "product": "ci",
        "age_condition": age_cond,
        "co_type": "all",
        "rate": rate,
        "source_label": f"CI CO {label}",
        "source_row": row,
        "notes": notes,
    })

# =============================================================================
# SP CO (rows 347-348) — shared across spouse_gla and spouse_lsdb
# Using product="spouse" to mirror how CI shares via bare "ci" key.
# =============================================================================
SP_CO = [
    (347, "<=65", "SP CO rate: age ≤65 (shared spouse_gla & spouse_lsdb)"),
    (348, ">65",  "SP CO rate: age >65 (shared spouse_gla & spouse_lsdb)"),
]

for row, age_cond, notes in SP_CO:
    label = ws.cell(row=row, column=5).value  # col E
    rate  = ws.cell(row=row, column=6).value  # col F
    all_rows.append({
        "product": "spouse",
        "age_condition": age_cond,
        "co_type": "all",
        "rate": rate,
        "source_label": f"SP CO {label}",
        "source_row": row,
        "notes": notes,
    })

print(f"Total rows harvested: {len(all_rows)}")
print(f"\nAll rows preview:")
for r in all_rows:
    print(f"  {r['product']:8s} | {r['age_condition']:9s} | {r['co_type']:26s} | "
          f"rate={r['rate']} ({type(r['rate']).__name__})")

Total rows harvested: 11

All rows preview:
  gla      | <=65      | retirement_and_withdrawal  | rate=0.037 (float)
  gla      | <=65      | withdrawal_only            | rate=0.02 (float)
  gla      | >65       | retirement_and_withdrawal  | rate=0.037 (float)
  gla      | >65       | withdrawal_only            | rate=0.02 (float)
  gla      | code_map  | retirement_and_withdrawal  | rate=1 (int)
  gla      | code_map  | withdrawal_only            | rate=2 (int)
  lsdb     | <=65      | all                        | rate=0.05 (float)
  lsdb     | >65       | all                        | rate=0.05 (float)
  ci       | <=65      | all                        | rate=0.05 (float)
  spouse   | <=65      | all                        | rate=0.037 (float)
  spouse   | >65       | all                        | rate=0.037 (float)


In [4]:
def classify(v) -> str:
    if v is None:            return "null"
    if isinstance(v, bool):  return "bool"
    if isinstance(v, int):   return "int"
    if isinstance(v, float): return "float"
    return "string"

for r in all_rows:
    v = r["rate"]
    r["param_type"] = classify(v)
    r["rate_str"] = "" if v is None else str(v)

df = pl.DataFrame(all_rows).select([
    "product",
    "age_condition",
    "co_type",
    pl.col("rate_str").alias("rate"),
    "param_type",
    "source_label",
    "source_row",
    "notes",
])

# =============================================================================
# Structural validation
# =============================================================================
key_cols = ["product", "age_condition", "co_type"]
dupes = df.group_by(key_cols).len().filter(pl.col("len") > 1)
assert dupes.height == 0, f"Duplicate keys:\n{dupes}"

expected_products = {"gla", "lsdb", "ci", "spouse"}
actual_products = set(df["product"].unique().to_list())
missing = expected_products - actual_products
unexpected = actual_products - expected_products
assert not missing, f"Missing: {missing}"
assert not unexpected, f"Unexpected: {unexpected}"

# =============================================================================
# Frozen value expectations for A2026_07
# =============================================================================
def get_rate(product, age_cond, co_type):
    row = df.filter(
        (pl.col("product") == product)
        & (pl.col("age_condition") == age_cond)
        & (pl.col("co_type") == co_type)
    )
    assert row.height == 1, f"Missing: {product}/{age_cond}/{co_type}"
    return float(row["rate"][0])

frozen = [
    ("gla",    "<=65",     "retirement_and_withdrawal", 0.037),
    ("gla",    "<=65",     "withdrawal_only",           0.02),
    ("gla",    ">65",      "retirement_and_withdrawal", 0.037),
    ("gla",    ">65",      "withdrawal_only",           0.02),
    ("gla",    "code_map", "retirement_and_withdrawal", 1),
    ("gla",    "code_map", "withdrawal_only",           2),
    ("lsdb",   "<=65",     "all",                       0.05),
    ("lsdb",   ">65",      "all",                       0.05),
    ("ci",     "<=65",     "all",                       0.05),
    ("spouse", "<=65",     "all",                       0.037),
    ("spouse", ">65",      "all",                       0.037),
]
for product, age_cond, co_type, expected in frozen:
    actual = get_rate(product, age_cond, co_type)
    assert abs(actual - expected) < 1e-9, (
        f"{product}/{age_cond}/{co_type}: got {actual}, expected {expected}"
    )

print(f"✓ {df.height} rows validated")
print(f"✓ {len(expected_products)} products present: {sorted(actual_products)}")
print(f"✓ {len(frozen)} frozen value checks passed")

print(f"\nType distribution:")
print(df.group_by("param_type").len().sort("param_type"))

print(f"\nFinal DataFrame:")
print(df)

✓ 11 rows validated
✓ 4 products present: ['ci', 'gla', 'lsdb', 'spouse']
✓ 11 frozen value checks passed

Type distribution:
shape: (2, 2)
┌────────────┬─────┐
│ param_type ┆ len │
│ ---        ┆ --- │
│ str        ┆ u32 │
╞════════════╪═════╡
│ float      ┆ 9   │
│ int        ┆ 2   │
└────────────┴─────┘

Final DataFrame:
shape: (11, 8)
┌─────────┬─────────────┬─────────────┬───────┬────────────┬─────────────┬────────────┬────────────┐
│ product ┆ age_conditi ┆ co_type     ┆ rate  ┆ param_type ┆ source_labe ┆ source_row ┆ notes      │
│ ---     ┆ on          ┆ ---         ┆ ---   ┆ ---        ┆ l           ┆ ---        ┆ ---        │
│ str     ┆ ---         ┆ str         ┆ str   ┆ str        ┆ ---         ┆ i64        ┆ str        │
│         ┆ str         ┆             ┆       ┆            ┆ str         ┆            ┆            │
╞═════════╪═════════════╪═════════════╪═══════╪════════════╪═════════════╪════════════╪════════════╡
│ gla     ┆ <=65        ┆ retirement_ ┆ 0.037 ┆ float

In [5]:
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name}")
print(f"  Size:    {PARQUET_SIZE_KB:.2f} KB")
print(f"  SHA-256: {PARQUET_SHA[:16]}...")

master_wb = openpyxl.load_workbook(MASTER_XLSX)
if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated: {MASTER_XLSX.name}")

✓ Wrote conversion_option.parquet
  Size:    3.78 KB
  SHA-256: 32283d245409df6e...
✓ Master workbook updated: assumption_master_A2026_07.xlsx


In [6]:
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "assumption_version", "notes",
]

log_row = {
    "built_at": BUILT_AT,
    "built_by": BUILT_BY,
    "sheet": SHEET_NAME,
    "rows": df.height,
    "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}",
    "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name,
    "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA,
    "assumption_version": ASSUMPTION_VERSION,
    "notes": (
        "Conversion Option rates and structure. Products: gla (2x2 grid + code map), "
        "lsdb (2 rows), ci (1 row, no >65 in source), spouse (2 rows, shared across "
        "spouse_gla/spouse_lsdb). CI has no >65 row in source — preserved verbatim (rule 5.5)."
    ),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated")
print(f"\n{'='*60}")
print(f"BUILD COMPLETE: {SHEET_NAME}")
print(f"{'='*60}")

✓ build_log.csv updated

BUILD COMPLETE: conversion_option
